# Informer — Multi-Source Cotton Yield Estimation (Türkiye)

This notebook trains and evaluates a **Transformer-based Informer**
model for cotton yield estimation from multivariate time-series (MTS)
Earth Observation data, following the experimental setup described in
the paper *"xLSTM for Multi-Source Cotton Yield Estimation and Temporal
Interpretability Across Agro-Ecological Regions in Türkiye"*
(Advances in Space Research).

Informer is an efficient Transformer for long-sequence time-series,
using **ProbSparse self-attention**, **self-attention distilling**, and
a **generative-style decoder**. Here it serves as a contemporary
attention-based baseline against the recurrent models (LSTM, BiLSTM,
xLSTM). The implementation uses the official Informer2020 repository.

**Input data (C = 20 variables / time step):**
- Sentinel-1 SAR backscatter (VV, VH)
- Sentinel-2 optical (EVI)
- ERA5-Land reanalysis (temperatures, water content, precipitation,
  evaporation, solar radiation, ...)
- SoilGrids static covariates (sand, silt, clay, bulk density, ...)

Informer additionally uses **temporal marks** (year / month / month-half)
as `x_mark` inputs for its time-feature embedding.

**Temporal setup:** bi-weekly (early/late) intervals over **June–October**,
sequence length **T = 10**.

**Target:** commune-level statistical yield (kg/da, TUIK).

**Pipeline of this notebook**
1. Environment & imports (incl. cloning Informer2020)
2. Data loading, temporal features, indexing
3. Standardization (z-score)
4. MTS windowing -> `X (N, T, C)`, `y (N,)`, `X_time (N, T, 3)`
5. Train / validation / test split (**70 / 10 / 20**)
6. Informer regression model definition
7. Hyperparameter search reference (Optuna — see Table 2 of the paper)
8. Training with early stopping
9. Evaluation (R2, MAE, RMSE, MAPE)
10. Gradient-based temporal interpretability (saliency)


## 1. Environment & Imports

The experiments in the paper were run on an **NVIDIA T4 GPU (15 GB)**.
If you are running on Google Colab, mount Drive to access the dataset;
otherwise set `file_path` below to your local CSV.

The Informer model is provided by the official **Informer2020**
repository, cloned below.

In [ ]:
# (Colab only) mount Google Drive to access the dataset.
# Comment out if running locally.
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ModuleNotFoundError:
    print("Not running on Colab — skipping Drive mount.")

In [ ]:
!git clone https://github.com/zhouhaoyi/Informer2020.git

In [ ]:
%cd /content/Informer2020

In [ ]:
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm import tqdm

from models.model import Informer  # from the cloned Informer2020 repo

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
Rando = 42  # global random seed

## 2. Load Data & Temporal Features

The CSV holds one row per field per time step, with the multivariate
features and the `YIELD` target. Adjust `file_path` to point to the
dataset (publicly available at the project repository).

Informer uses temporal marks, so `year`, `month`, and `month_half`
(early/late) are derived from `month_period`.

In [ ]:
file_path = '/content/gdrive/MyDrive/EsraHoca/aggregated_15_5_ege_az_ay.csv'  # set your path
data = pd.read_csv(file_path)
print("Raw shape:", data.shape)

In [ ]:
# Extract temporal features
data['month_half'] = data['month_period'].apply(lambda x: 0 if 'first_half' in x else 1)
data['year'] = data['month_period'].apply(lambda x: int(x.split('-')[0]))
data['month'] = data['month_period'].apply(lambda x: int(x.split('-')[1]))


In [ ]:
data.columns

In [ ]:
data.head(2)

In [ ]:
data["REGION_ID"].unique()  # 1: Aegean, 2: Mediterranean, 3: Southeastern Anatolia

Set a composite index so the feature matrix contains only the
predictor columns, the temporal marks, and the target.

In [ ]:
data.set_index(['FieldId', 'month_period', "DistrictName", "REGION_ID", "day"], inplace=True)
data.head(5)

In [ ]:
data.shape

## 3. Standardization

All variables (including the target `YIELD`) are z-score standardized
with a single `StandardScaler`. The scaler statistics for the `YIELD`
column are stored so predictions can be inverse-transformed back to
kg/da for reporting.

In [ ]:
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)
data_norm = pd.DataFrame(data_scaled, columns=data.columns)
data_norm.head(2)

## 4. Build Multivariate Time-Series Windows

Each field is converted into a non-overlapping window of length
`WINDOW_SIZE = 10` (the bi-weekly June–October sequence, T = 10). The
yield label is taken from the last step of each window. The temporal
marks (`year`, `month`, `month_half`) are returned separately as
`X_time` for Informer's time-feature embedding.

Output shapes: `X = (N, T, C)`, `y = (N,)`, `X_time = (N, T, 3)`.

In [ ]:
def df_to_X_y_non_overlap(df, window_size=10):
    X, y, X_time = [], [], []
    time_features = df[['year', 'month', 'month_half']].values
    df_values = df.drop(columns=['YIELD', 'year', 'month', 'month_half']).values
    yield_values = df['YIELD'].values

    for i in range(0, len(df_values) - window_size + 1, window_size):
        X.append(df_values[i:i+window_size])
        y.append(yield_values[i+window_size-1])
        X_time.append(time_features[i:i+window_size])

    return np.array(X), np.array(y), np.array(X_time)

In [ ]:
WINDOW_SIZE = 10
X, y, X_time = df_to_X_y_non_overlap(data_norm, WINDOW_SIZE)
print("X:", X.shape, "| y:", y.shape, "| X_time:", X_time.shape)

## 5. Dataset & DataLoader

The dataset returns a `(features, temporal_marks, label)` triple so the
Informer encoder/decoder can consume the time features.

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, X_time):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.X_time = torch.tensor(X_time, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.X_time[idx], self.y[idx]

## 6. Train / Validation / Test Split

Following the paper: **70% training, 10% validation, 20% test**
(`random_state=Rando`). The 20% test split is held out first; the
remaining 80% is split again so that validation is 10% of the total
(0.125 x 0.8 = 0.10). The temporal marks `X_time` are split alongside.

In [ ]:
# 20% test
X_train, X_test, y_train, y_test, X_time_train, X_time_test = train_test_split(
    X, y, X_time, test_size=0.20, random_state=Rando)

# 10% validation of the total (0.125 of the remaining 80%)
X_train, X_val, y_train, y_val, X_time_train, X_time_val = train_test_split(
    X_train, y_train, X_time_train, test_size=0.125, random_state=Rando)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
batch_size = 32

train_loader = DataLoader(TimeSeriesDataset(X_train, y_train, X_time_train),
                          batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TimeSeriesDataset(X_val,   y_val,   X_time_val),
                          batch_size=batch_size)
test_loader  = DataLoader(TimeSeriesDataset(X_test,  y_test,  X_time_test),
                          batch_size=batch_size)

## 7. Informer Regression Model

`YieldPredictionModel` wraps the Informer encoder-decoder and adds a
linear regression head over the flattened decoder output. The Informer
is configured with `d_model=128`, `n_heads=4`, `e_layers=2`,
`d_layers=1`, `d_ff=256`, ProbSparse attention (`attn='prob'`),
time-feature embedding (`embed='timeF'`), distilling, and a GELU
activation — consistent with Table 2 of the paper.

The model carries its own training and evaluation loops with:

- **Loss:** MSE
- **Optimizer:** Adam
- **Early stopping** on validation R2 (with `min_epochs` and `patience`)
- Logging of train/val loss, train/val R2, MAE, RMSE, MAPE, gradient norm

`inverse_transform_yield` maps standardized predictions back to kg/da
using the stored `YIELD` scaler statistics.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from models.model import Informer  # Informer modelini içe aktarıyoruz.
import matplotlib.pyplot as plt

# ----------------------------
# YieldPredictionModel tanımı
# ----------------------------
class YieldPredictionModel(nn.Module):
    @staticmethod
    def weight_init(m):
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            if m.weight is not None:
                nn.init.kaiming_normal_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def __init__(self, enc_in, dec_in, c_out, seq_len, label_len, pred_len, lr=1e-4):
        """
        - enc_in, dec_in: encoder ve decoder giriş özellik sayısı
        - c_out: Informer modelinin çıktı kanallarının sayısı (örn. 3)
        - seq_len: giriş dizisinin uzunluğu
        - label_len: decoder girişindeki etiket uzunluğu
        - pred_len: tahmin edilecek adım sayısı
        """
        super().__init__()
        self.informer = Informer(
            enc_in=enc_in, dec_in=dec_in, c_out=c_out,
            seq_len=seq_len, label_len=label_len, out_len=pred_len,
            d_model=128, n_heads=4, e_layers=2, d_layers=1, d_ff=256,
            dropout=0.1, attn='prob', embed='timeF', freq='h', activation='gelu',
            output_attention=False, distil=True, mix=True,
            device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
        )
        # Regresyon için çıkışı tek nöronla elde ediyoruz.
        self.regressor = nn.Linear(c_out * pred_len, 1)

        self.apply(self.weight_init)
        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(self.parameters(), lr=lr)

        # Loglar
        self.gradient_norms = []
        self.losses = []          # Eğitim kaybı
        self.train_r2_log = []    # Eğitim R²
        self.val_losses = []      # Doğrulama kaybı
        self.val_r2_log = []      # Doğrulama R²
        self.mae_log = []
        self.rmse_log = []
        self.mape_log = []

        # Model konfigürasyonları
        self.seq_len = seq_len
        self.label_len = label_len
        self.pred_len = pred_len
        self.enc_in = enc_in
        self.dec_in = dec_in

    def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec):
        informer_out = self.informer(x_enc, x_mark_enc, x_dec, x_mark_dec)
        informer_flat = informer_out.reshape(informer_out.size(0), -1)
        return self.regressor(informer_flat).squeeze(-1)  # <-- SADECE SON BOYUTU squeeze edin.


    def evaluate_loader(self, loader, device, epoch, calculate_metrics=True):
        """
        Verilen loader (eğitim/validasyon/test) üzerinde ileri geçiş yapıp,
        loss ve (varsa) diğer metrikleri (MAE, RMSE, MAPE, R²) hesaplar.
        Eğitim sırasında time özellikleri mevcutsa onları kullanır, yoksa sıfır tensörler oluşturur.
        """
        self.eval()
        running_loss = 0.0
        total_samples = 0
        predictions = []
        actuals = []
        with torch.no_grad():
            for batch in tqdm(loader, desc=f"Eval Epoch {epoch + 1}", leave=False):
                # Eğer batch içinde zaman özellikleri varsa
                if len(batch) == 3:
                    inputs, time_feats, labels = batch
                    inputs = inputs.to(device, dtype=torch.float32)
                    labels = labels.to(device, dtype=torch.float32)
                    time_feats = time_feats.to(device, dtype=torch.float32)
                    # Eğer time_feats son boyut 3 ise 4. boyut oluşturuyoruz.
                    if time_feats.size(-1) == 3:
                        pad = torch.zeros(time_feats.size(0), time_feats.size(1), 1, device=device)
                        time_feats = torch.cat([time_feats, pad], dim=-1)
                    x_mark_enc = time_feats
                    batch_size = inputs.shape[0]
                    x_mark_dec = torch.zeros(batch_size, self.label_len + self.pred_len, time_feats.shape[-1]).to(device)
                    x_mark_dec[:, :self.label_len, :] = time_feats[:, -self.label_len:, :]
                else:
                    inputs, labels = batch
                    inputs = inputs.to(device, dtype=torch.float32)
                    labels = labels.to(device, dtype=torch.float32)
                    batch_size = inputs.shape[0]
                    x_mark_enc = torch.zeros(batch_size, self.seq_len, 4).to(device)
                    x_mark_dec = torch.zeros(batch_size, self.label_len + self.pred_len, 4).to(device)

                # x_dec oluşturulması (decoder için giriş)
                batch_size = inputs.shape[0]
                x_dec = torch.zeros(batch_size, self.label_len + self.pred_len, self.dec_in).to(device)
                x_dec[:, :self.label_len, :] = inputs[:, -self.label_len:, :]

                outputs = self.forward(inputs, x_mark_enc, x_dec, x_mark_dec).squeeze()
                loss = self.criterion(outputs, labels)
                running_loss += loss.item() * inputs.size(0)
                total_samples += inputs.size(0)

                # Çıktıları ters ölçeklendiriyoruz
                outputs_np = inverse_transform_yield(outputs.cpu().numpy())
                labels_np = inverse_transform_yield(labels.cpu().numpy())
                predictions.extend(outputs_np)
                actuals.extend(labels_np)

        loss_val = running_loss / total_samples
        predictions = np.array(predictions)
        actuals = np.array(actuals)
        if calculate_metrics:
            mae = mean_absolute_error(actuals, predictions)
            rmse = np.sqrt(mean_squared_error(actuals, predictions))
            mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100
            r2 = r2_score(actuals, predictions)
            return loss_val, mae, rmse, mape, r2
        else:
            r2 = r2_score(actuals, predictions)
            return r2

    def train_model(self, train_loader, val_loader, test_loader=None, epochs=20, min_epochs=10, patience=5):
        """
        Modeli eğitirken; her epoch sonunda:
         - Eğitim kaybı ve R² metriği hesaplanır.
         - Doğrulama verisi üzerinde loss, MAE, RMSE, MAPE, R² hesaplanır.
         - Doğrulama R² değeri iyileşmezse early stopping uygulanır.
        """
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.to(device)

        best_val_r2 = float('-inf')
        epochs_no_improve = 0
        loss_oscillations = []

        for epoch in range(epochs):
            self.train()
            running_loss = 0.0
            total_samples = 0
            epoch_gradient_norm = []
            train_preds = []
            train_labels = []

            for inputs, time_feats, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}"):
                inputs = inputs.to(device, dtype=torch.float32)
                labels = labels.to(device, dtype=torch.float32)
                time_feats = time_feats.to(device, dtype=torch.float32)

                # Eğer time_feats son boyutu 3 ise 4. boyuta genişletiyoruz.
                if time_feats.size(-1) == 3:
                    pad = torch.zeros(time_feats.size(0), time_feats.size(1), 1, device=device)
                    time_feats = torch.cat([time_feats, pad], dim=-1)

                batch_size = inputs.shape[0]
                x_mark_enc = time_feats
                x_mark_dec = torch.zeros(batch_size, self.label_len + self.pred_len, time_feats.shape[-1]).to(device)
                x_mark_dec[:, :self.label_len, :] = time_feats[:, -self.label_len:, :]

                x_dec = torch.zeros(batch_size, self.label_len + self.pred_len, self.dec_in).to(device)
                x_dec[:, :self.label_len, :] = inputs[:, -self.label_len:, :]

                self.optimizer.zero_grad()
                outputs = self.forward(inputs, x_mark_enc, x_dec, x_mark_dec).squeeze()
                loss = self.criterion(outputs, labels)
                loss.backward()

                # Gradient clipping
                #torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=1.0)

                total_norm = 0
                for p in self.parameters():
                    if p.grad is not None:
                        param_norm = p.grad.data.norm(2)
                        total_norm += param_norm.item() ** 2
                total_norm = total_norm ** 0.5
                epoch_gradient_norm.append(total_norm)

                self.optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                total_samples += inputs.size(0)

                # Eğitim metrikleri için tahmin ve etiketleri kaydediyoruz.
                outputs_np = inverse_transform_yield(outputs.cpu().detach().numpy())
                labels_np = inverse_transform_yield(labels.cpu().detach().numpy())
                train_preds.extend(outputs_np)
                train_labels.extend(labels_np)

            mean_gradient_norm = np.mean(epoch_gradient_norm)
            self.gradient_norms.append(mean_gradient_norm)
            epoch_loss = running_loss / total_samples
            self.losses.append(epoch_loss)

            if epoch > 0:
                loss_oscillation = abs(self.losses[-1] - self.losses[-2])
                loss_oscillations.append(loss_oscillation)

            # Eğitim verisi üzerinden R² hesaplama
            train_r2 = r2_score(np.array(train_labels), np.array(train_preds))
            self.train_r2_log.append(train_r2)

            # Doğrulama verisi değerlendirmesi
            val_loss, mae, rmse, mape, val_r2 = self.evaluate_loader(val_loader, device, epoch, calculate_metrics=True)
            self.val_losses.append(val_loss)
            self.val_r2_log.append(val_r2)
            self.mae_log.append(mae)
            self.rmse_log.append(rmse)
            self.mape_log.append(mape)

            print(f'Epoch: {epoch + 1}, Train Loss: {epoch_loss:.4f}, Train R²: {train_r2:.4f}, '
                  f'Val Loss: {val_loss:.4f}, Val R²: {val_r2:.4f}, MAE: {mae:.2f}, RMSE: {rmse:.2f}, MAPE: {mape:.2f}%')

            if val_r2 > best_val_r2:
                best_val_r2 = val_r2
                epochs_no_improve = 0
                torch.save(self.state_dict(), 'best_model.pth')
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience and epoch >= min_epochs:
                print("Early stopping triggered")
                break

        self.load_state_dict(torch.load('best_model.pth'))

        if test_loader:
            test_r2 = self.evaluate_loader(test_loader, device, epoch=epochs, calculate_metrics=False)
            print(f'Final Test R²: {test_r2:.4f}')

        return {
            "epoch_loss": self.losses,
            "train_r2": self.train_r2_log,
            "val_loss": self.val_losses,
            "val_r2": self.val_r2_log,
            "mae": self.mae_log,
            "rmse": self.rmse_log,
            "mape": self.mape_log,
            "gradient_norm": self.gradient_norms,
            "loss_oscillations": loss_oscillations
        }
    def test_model(self, loader, device, epoch, calculate_metrics=True):
        self.eval()
        running_loss = 0.0
        total_samples = 0
        predictions = []
        actuals = []

        with torch.no_grad():
            for batch in tqdm(loader, desc=f"Test Epoch {epoch + 1}", leave=False):
                if len(batch) == 2:
                    inputs, labels = batch
                else:
                    inputs, _, labels = batch

                inputs = inputs.to(device, dtype=torch.float32)
                labels = labels.to(device, dtype=torch.float32)
                batch_size = inputs.shape[0]

                x_mark_enc = torch.zeros(batch_size, self.seq_len, 4).to(device)
                x_mark_dec = torch.zeros(batch_size, self.label_len + self.pred_len, 4).to(device)
                x_dec = torch.zeros(batch_size, self.label_len + self.pred_len, self.dec_in).to(device)
                x_dec[:, :self.label_len, :] = inputs[:, -self.label_len:, :]

                outputs = self.forward(inputs, x_mark_enc, x_dec, x_mark_dec).squeeze()
                loss = self.criterion(outputs, labels)

                running_loss += loss.item() * inputs.size(0)
                total_samples += inputs.size(0)

                outputs = inverse_transform_yield(outputs.cpu().numpy())
                labels = inverse_transform_yield(labels.cpu().numpy())

                predictions.extend(outputs)
                actuals.extend(labels)

        loss_val = running_loss / total_samples
        predictions = np.array(predictions)
        actuals = np.array(actuals)

        if calculate_metrics:
            mae = mean_absolute_error(actuals, predictions)
            rmse = np.sqrt(mean_squared_error(actuals, predictions))
            mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100
            r2 = r2_score(actuals, predictions)
            return loss_val, mae, rmse, mape, r2
        else:
            r2 = r2_score(actuals, predictions)
            return r2

# ---------------------------------------------------
# inverse_transform_yield fonksiyonu: scaler kullanarak ters dönüşüm
# scaler nesnesi ve data DataFrame'inin tanımlı olması gerekir.
# ---------------------------------------------------
yield_index = list(data.columns).index('YIELD')

def inverse_transform_yield(yield_scaled):
    yield_scaled = np.array(yield_scaled)
    yield_original = yield_scaled * scaler.scale_[yield_index] + scaler.mean_[yield_index]
    return yield_original


## 8. Hyperparameter Search (Optuna) — Reference

Hyperparameters were tuned with **Optuna** using **5-fold
cross-validation**, maximizing the **Concordance Correlation
Coefficient (CCC)**. The search space and the optimal configuration
selected for Informer (Table 2 of the paper) are:

| Hyperparameter                 | Tested range                 | Optimal  |
|--------------------------------|------------------------------|----------|
| Model dimension (d_model)      | {64, 128, 256}               | **128**  |
| Feed-forward dimension (d_ff)  | {128, 256, 512}              | **256**  |
| Learning rate (Adam)           | [1e-5, 1e-3] (log-uniform)   | **3.75e-4** |

The d_model / d_ff optimal values are already set inside the model
definition above. The search itself is **not re-run by default** (it is
expensive); the optimal values are applied directly in Section 9.

In [ ]:
RUN_OPTUNA = False  # set True to re-run the hyperparameter search (slow)

if RUN_OPTUNA:
    import optuna

    def concordance_correlation_coefficient(x, y):
        """Concordance Correlation Coefficient (CCC)."""
        if x.ndim == 1:
            x = x[:, np.newaxis]
        if y.ndim == 1:
            y = y[:, np.newaxis]
        sxy = np.sum(np.dot((x - x.mean())[:, 0], (y - y.mean())[:, 0])) / x.shape[0]
        rhoc = 2 * sxy / (np.var(x) + np.var(y) + (x.mean() - y.mean()) ** 2)
        return rhoc

    def objective(trial):
        d_model = trial.suggest_categorical('d_model', [64, 128, 256])
        d_ff = trial.suggest_categorical('d_ff', [128, 256, 512])
        lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)

        pred_len = 1 if len(y_train.shape) == 1 else y_train.shape[1]
        model = YieldPredictionModel(
            enc_in=X.shape[-1], dec_in=X.shape[-1], c_out=3,
            seq_len=10, label_len=5, pred_len=pred_len, lr=lr,
        )
        model.train_model(train_loader, val_loader, epochs=100, min_epochs=10, patience=5)
        return max(model.val_r2_log)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=10)
    print('Best trial:', study.best_trial.params)

## 9. Train the Informer (optimal configuration)

Trained with the Optuna-selected optimal hyperparameters from Table 2:
`d_model=128`, `d_ff=256` (set in the model definition), `lr=3.75e-4`.
Early stopping monitors validation R2. The best checkpoint (by
validation R2) is saved to `best_model.pth` and reloaded at the end of
training.

In [ ]:
# pred_len = 1 for a single scalar yield target
pred_len = 1 if len(y_train.shape) == 1 else y_train.shape[1]

net = YieldPredictionModel(
    enc_in=X.shape[-1],
    dec_in=X.shape[-1],
    c_out=3,           # Informer output channels
    seq_len=10,
    label_len=5,
    pred_len=pred_len,
    lr=3.75e-4,        # Optuna optimal
)
net.train()

train_results = net.train_model(train_loader, val_loader, epochs=250, patience=25)

# Test-set R2 (full metrics computed in Section 10)
test_r2 = net.evaluate_loader(test_loader, device, epoch=250, calculate_metrics=False)
print(f'Final Test R^2: {test_r2:.4f}')

Optionally reload the best checkpoint explicitly before evaluation.

In [ ]:
# net = YieldPredictionModel(enc_in=X.shape[-1], dec_in=X.shape[-1], c_out=3,
#                            seq_len=10, label_len=5, pred_len=pred_len, lr=3.75e-4)
# net.load_state_dict(torch.load('best_model.pth'))
# net = net.to(device)

### Training curves (loss / R2 / gradient norm)

In [ ]:
plt.figure(figsize=(18, 5))

# Loss grafiği: Eğitim ve Validasyon
plt.subplot(1, 3, 1)
plt.plot(train_results["epoch_loss"], label="Train Loss")
plt.plot(train_results["val_loss"], label="Validation Loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# R² grafiği: Eğitim ve Validasyon
plt.subplot(1, 3, 2)
plt.plot(train_results["train_r2"], label="Train R²")
plt.plot(train_results["val_r2"], label="Validation R²")
plt.title("R²")
plt.xlabel("Epoch")
plt.ylabel("R²")
plt.legend()

# Gradient Norm grafiği
plt.subplot(1, 3, 3)
plt.plot(train_results["gradient_norm"], label="Gradient Norm")
plt.title("Gradient Norm")
plt.xlabel("Epoch")
plt.ylabel("Norm")
plt.legend()

plt.tight_layout()
plt.show()


## 10. Evaluation on the Test Set

Predictions are inverse-transformed to kg/da before computing
**R2, MAE, RMSE, and MAPE**. The evaluation builds the decoder inputs
and temporal marks the same way as during training.

In [ ]:
net.eval()
y_pred, y_test = [], []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with torch.no_grad():
    for data_load in test_loader:
        if len(data_load) == 3:
            inputs, time_feats, labels = data_load

            inputs = inputs.to(device, dtype=torch.float32)
            time_feats = time_feats.to(device, dtype=torch.float32)
            labels = labels.to(device, dtype=torch.float32)

            # Eğer time_feats son boyut 3 ise 4. boyut ekleyelim
            if time_feats.size(-1) == 3:
                pad = torch.zeros(time_feats.size(0), time_feats.size(1), 1, device=device)
                time_feats = torch.cat([time_feats, pad], dim=-1)

            x_mark_enc = time_feats
            x_mark_dec = torch.zeros(inputs.size(0), net.label_len + net.pred_len, time_feats.shape[-1]).to(device)
            x_mark_dec[:, :net.label_len, :] = time_feats[:, -net.label_len:, :]

        else:
            inputs, labels = data_load
            inputs = inputs.to(device, dtype=torch.float32)
            labels = labels.to(device, dtype=torch.float32)

            x_mark_enc = torch.zeros(inputs.size(0), net.seq_len, 4).to(device)
            x_mark_dec = torch.zeros(inputs.size(0), net.label_len + net.pred_len, 4).to(device)

        x_dec = torch.zeros(inputs.size(0), net.label_len + net.pred_len, net.dec_in).to(device)
        x_dec[:, :net.label_len, :] = inputs[:, -net.label_len:, :]

        outputs = net(inputs, x_mark_enc, x_dec, x_mark_dec)
        outputs = outputs.squeeze().cpu().numpy()
        labels = labels.cpu().numpy()

        y_pred.extend(outputs)
        y_test.extend(labels)

# Ters dönüşüm
y_pred = inverse_transform_yield(np.array(y_pred))
y_test = inverse_transform_yield(np.array(y_test))

# Metrikler
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
r2 = r2_score(y_test, y_pred)

print(f'Final Test - MAE: {mae:.3f}, RMSE: {rmse:.3f}, MAPE: {mape:.3f}%, R^2: {r2:.3f}')


## 11. Gradient-Based Temporal Interpretability

Interpretability uses **gradient-based saliency**: the gradient of the
predicted yield with respect to each input is computed on the test set
to reveal which **time intervals** and **features** the model relies on.
For Informer, the required decoder inputs and (zero) temporal marks are
constructed before the backward pass.

The temporal axis spans the study season from **early June (June_1)**
to **late October (October_2)**, consistent with the cotton phenology
described in the paper. The saliency array is saved as `.npy` for the
cross-model comparison figures (LSTM vs. BiLSTM vs. xLSTM vs. Informer).

In [ ]:
# Compute input-gradient saliency on the test set.
net = net.to(device)

test_input = torch.tensor(X_test, dtype=torch.float32).to(device)
test_input.requires_grad = True

# Informer-specific decoder inputs and (zero) temporal marks.
batch_size = test_input.shape[0]
x_mark_enc = torch.zeros(batch_size, net.seq_len, 4, device=device)
x_mark_dec = torch.zeros(batch_size, net.label_len + net.pred_len, 4, device=device)
x_dec = torch.zeros(batch_size, net.label_len + net.pred_len, net.dec_in, device=device)
x_dec[:, :net.label_len, :] = test_input[:, -net.label_len:, :]

net.zero_grad()
net.train()  # match training-mode behaviour used when computing saliency
outputs = net(test_input, x_mark_enc, x_dec, x_mark_dec)
loss = outputs.mean()
loss.backward()

saliency = test_input.grad.data.cpu().numpy()   # (N, T, C)
abs_saliency = np.abs(saliency)
mean_saliency = abs_saliency.mean(axis=0)        # (T, C)

# Bi-weekly intervals, June-October (T = 10)
months = ["June_1", "June_2", "July_1", "July_2", "August_1",
          "August_2", "September_1", "September_2", "October_1", "October_2"]

# Absolute saliency heatmap (time step x feature)
plt.figure(figsize=(12, 8))
plt.imshow(mean_saliency.T, aspect='auto', cmap='jet')
plt.colorbar(label='Absolute saliency')
plt.xlabel('Time step')
plt.ylabel('Input feature')
plt.title('Informer Saliency Map (test set)')
plt.xticks(range(len(months)), months, rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Total absolute saliency across time steps
plt.figure(figsize=(12, 6))
plt.plot(mean_saliency.sum(axis=1), marker='o')
plt.xlabel('Time step')
plt.ylabel('Total absolute saliency')
plt.title('Informer Total Saliency Across Time Steps')
plt.xticks(range(len(months)), months, rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Save saliency for the cross-model temporal-importance comparison figures.
np.save(f'saliency_informer_all_{int(Rando)}.npy', saliency)